# Open List Proportional Representation Demo

**Interactive demonstration of how voters can override party rankings**

This notebook shows the two-stage process:
1. **Stage 1**: Allocate seats to parties (using D'Hondt or Sainte-Laguë)
2. **Stage 2**: Determine which candidates win those seats (based on preference votes)

In [ ]:
# Import the classes
import sys
sys.path.append('src')

from main import Candidate, OpenListPR

## Scenario: Three Parties Competing for 10 Seats

**Party Votes** (determines seat allocation):
- Party A: 100,000 votes
- Party B: 80,000 votes  
- Party C: 30,000 votes

**Key Question**: Which specific candidates from each party win seats?

In [ ]:
# Define the candidates with their party-assigned list positions
candidates = [
    # Party A candidates (list positions 1-5)
    Candidate(name="Alice", party="Party A", list_position=1),
    Candidate(name="Bob", party="Party A", list_position=2),
    Candidate(name="Carol", party="Party A", list_position=3),
    Candidate(name="David", party="Party A", list_position=4),
    Candidate(name="Eve", party="Party A", list_position=5),
    
    # Party B candidates (list positions 1-4)
    Candidate(name="Frank", party="Party B", list_position=1),
    Candidate(name="Grace", party="Party B", list_position=2),
    Candidate(name="Henry", party="Party B", list_position=3),
    Candidate(name="Iris", party="Party B", list_position=4),
    
    # Party C candidates (list positions 1-3)
    Candidate(name="Jack", party="Party C", list_position=1),
    Candidate(name="Kate", party="Party C", list_position=2),
    Candidate(name="Leo", party="Party C", list_position=3),
]

print("Candidates created!")
print(f"Total candidates: {len(candidates)}")

In [ ]:
# Party votes (Stage 1 input)
party_votes = {
    'Party A': 100000,
    'Party B': 80000,
    'Party C': 30000,
}

# Candidate preference votes (Stage 2 input)
# Notice: Some lower-ranked candidates have MORE preference votes!
candidate_votes = {
    # Party A: Bob (position 2) is most popular!
    "Alice": 8000,    # List position 1, but low votes
    "Bob": 35000,     # List position 2, but HIGH votes - will jump ahead!
    "Carol": 28000,   # List position 3, second highest votes
    "David": 15000,   # List position 4
    "Eve": 10000,     # List position 5
    
    # Party B: Grace (position 2) gets more votes than Frank!
    "Frank": 25000,   # List position 1
    "Grace": 30000,   # List position 2, but more votes - jumps ahead!
    "Henry": 15000,   # List position 3
    "Iris": 8000,     # List position 4
    
    # Party C: Kate is very popular!
    "Jack": 5000,     # List position 1
    "Kate": 20000,    # List position 2, but most votes - jumps ahead!
    "Leo": 3000,      # List position 3
}

print("Election data prepared!")

## Run the Election with D'Hondt Method

In [ ]:
# Create the election system
election = OpenListPR(
    parties=['Party A', 'Party B', 'Party C'],
    candidates=candidates,
    party_votes=party_votes,
    candidate_votes=candidate_votes,
    total_seats=10,
    method='dhondt',
    ranking_variant='pure'  # Pure open list: voters fully control ranking
)

# Run the election
results = election.run_election()

print("Election complete!")

## Results: Stage 1 - Party Seat Allocation

In [ ]:
print("=" * 70)
print("STAGE 1: PARTY SEAT ALLOCATION (D'Hondt Method)")
print("=" * 70)

total_votes = sum(party_votes.values())

for party, seats in results['party_seats'].items():
    votes = party_votes[party]
    vote_pct = (votes / total_votes) * 100
    seat_pct = (seats / 10) * 100
    print(f"{party:10s}: {seats:2d} seats ({seat_pct:5.1f}%) | {votes:,} votes ({vote_pct:5.1f}%)")

print("=" * 70)

## Results: Stage 2 - Elected Candidates

**Key insight**: Notice which candidates "jumped" ahead of their list position!

In [ ]:
print("\n" + "=" * 80)
print("STAGE 2: ELECTED CANDIDATES (sorted by preference votes)")
print("=" * 80)

for party in ['Party A', 'Party B', 'Party C']:
    print(f"\n{party}:")
    print(f"  {'Candidate':12s} {'List Pos':>10s} {'Pref Votes':>12s} {'Final Rank':>12s} {'Status':>10s}")
    print("  " + "-" * 68)
    
    # Get all candidates from this party, sorted by final rank
    party_candidates = [c for c in candidates if c._party == party]
    party_candidates.sort(key=lambda c: c.final_rank if c.final_rank else 999)
    
    for candidate in party_candidates:
        pref = candidate_votes.get(candidate._name, 0)
        status = "✓ ELECTED" if candidate.elected else ""
        rank = candidate.final_rank if candidate.final_rank else "-"
        
        # Mark candidates who jumped ahead of their list position
        jumped = ""
        if candidate.final_rank and candidate.final_rank < candidate._list_position:
            jumped = " ⬆️ JUMPED!"
        
        print(f"  {candidate._name:12s} {candidate._list_position:10d} {pref:12,} {str(rank):>12s} {status:>10s}{jumped}")

print("\n" + "=" * 80)

## Analysis: The Power of Voter Preferences

Let's highlight the candidates who jumped ahead of their party's ranking:

In [ ]:
print("\n🎯 CANDIDATES WHO JUMPED AHEAD OF THEIR LIST POSITION:")
print("=" * 70)

jumpers = []
for candidate in results['elected_candidates']:
    if candidate.final_rank < candidate._list_position:
        jumpers.append(candidate)
        party_seats = results['party_seats'][candidate._party]
        print(f"  • {candidate._name} ({candidate._party}):")
        print(f"      List position: #{candidate._list_position}")
        print(f"      Final rank: #{candidate.final_rank}")
        print(f"      Preference votes: {candidate.preference_votes:,}")
        print(f"      → Jumped {candidate._list_position - candidate.final_rank} position(s)!\n")

if not jumpers:
    print("  No candidates jumped ahead (list positions were respected)")
else:
    print(f"\nTotal jumpers: {len(jumpers)} out of {len(results['elected_candidates'])} elected")
    print(f"This demonstrates PURE OPEN LIST - voters override party rankings!")

## Comparison: What if it was a Closed List?

In a closed list system, parties control the order. Let's see the difference:

In [ ]:
print("\n📊 OPEN LIST vs CLOSED LIST COMPARISON")
print("=" * 70)

for party in ['Party A', 'Party B', 'Party C']:
    seats_won = results['party_seats'][party]
    print(f"\n{party} (won {seats_won} seats):")
    
    # Get candidates in list order
    party_candidates = [c for c in candidates if c._party == party]
    party_candidates.sort(key=lambda c: c._list_position)
    
    print(f"  {'':3s} {'Closed List (party order)':30s} {'Open List (voter order)':30s}")
    print("  " + "-" * 66)
    
    for i in range(seats_won):
        closed_winner = party_candidates[i]._name  # First N by list position
        
        # Find open list winner at this rank
        open_winner = ""
        for c in candidates:
            if c._party == party and c.final_rank == i + 1:
                open_winner = c._name
                break
        
        different = "" if closed_winner == open_winner else " ← DIFFERENT!"
        print(f"  #{i+1} {closed_winner:30s} {open_winner:30s}{different}")

## Try with Sainte-Laguë Method

Let's see if a different allocation method changes the results:

In [ ]:
# Run with Sainte-Laguë
election_sl = OpenListPR(
    parties=['Party A', 'Party B', 'Party C'],
    candidates=candidates,  # Reset candidates
    party_votes=party_votes,
    candidate_votes=candidate_votes,
    total_seats=10,
    method='satinelague',
    ranking_variant='pure'
)

# Reset candidate states
for c in candidates:
    c.elected = False
    c.final_rank = None
    c.preference_votes = 0

results_sl = election_sl.run_election()

print("\n" + "=" * 70)
print("COMPARISON: D'Hondt vs Sainte-Laguë Seat Allocation")
print("=" * 70)
print(f"{'Party':10s} {'D\'Hondt':>10s} {'Sainte-Laguë':>15s} {'Difference':>15s}")
print("-" * 70)

for party in ['Party A', 'Party B', 'Party C']:
    dhondt_seats = results['party_seats'][party]
    sl_seats = results_sl['party_seats'][party]
    diff = sl_seats - dhondt_seats
    diff_str = f"{diff:+d}" if diff != 0 else "same"
    print(f"{party:10s} {dhondt_seats:10d} {sl_seats:15d} {diff_str:>15s}")

print("\nNote: Sainte-Laguë tends to favor smaller parties slightly more than D'Hondt")

## Summary

**Key Takeaways:**

1. **Two-Stage Process**:
   - Stage 1: D'Hondt/Sainte-Laguë determines party seats
   - Stage 2: Preference votes determine candidate ranking

2. **Voter Power**:
   - In pure open list, voters can completely override party rankings
   - Popular candidates jump ahead regardless of list position

3. **Democratic Choice**:
   - Parties suggest rankings (list positions)
   - Voters make final decision (preference votes)
   - Most democratic form of proportional representation